Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import os
import shutil

# Step 1: Ensure 'data' directory exists
if not os.path.exists("data"):
    os.makedirs("data")

# Step 2: Run the SQL query and get the results as a DataFrame
query = """
with t as 
(select cast(json_extract(dt.metadata, '$.correlation_id') as varchar) AS chat_reference_id, 
                    result,
                    us.name as annotatedBy,
                    rs.annotated_at as annotatedAt,
                    dt.ticket_id as ticket_id
    from (
            select id,data_id,annotated_by,annotated_at,result,status 
            from crm_ntl_annotation.annotation_results
            where date(cast(split_part(replace(annotated_at, 'T', ' '), '.', 1) as timestamp) + interval '330' minute)>=date('2025-04-08')
            )  rs 
left join   (
            select id,uuid,data,mime_type,batch_id,ticket_id,template_id,metadata,created_at,created_by,version,kafka_state,assignment_preferences,modified_at,modified_by,is_active,ticket_uuid 
            from crm_ntl_annotation.annotation_data
            ) dt on rs.data_id =dt.id 
left join mjolnir_service.users us on us.account_id = rs.annotated_by
-- left join cdas_service.chat_orchestrator co on co.chat_reference_id = json_extract_scalar(dt.metadata, '$.correlation_id')
--left join collection_service.collection_cases cc on co.entity_Id=cc.account_number
where dt.template_id = 187),

parsed AS (
  SELECT chat_reference_id,annotatedAt,annotatedBy,result,
    cast(json_parse(result) as array(json)) as x
  FROM t
),


data as(
Select 
    chat_reference_id,annotatedBy,
    date("date_add"('minute', 330, "date_parse"("substring"("replace"(annotatedAT, 'T', ' '), 1, 19), '%Y-%m-%d %H:%i:%s'))) as annotated_at_date,
("date_add"('minute', 330, "date_parse"("substring"("replace"(annotatedAT, 'T', ' '), 1, 19), '%Y-%m-%d %H:%i:%s'))) as annotated_at_ist,
    max(case when json_extract_scalar(t.x,'$.from_name') = 'category' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Category",
    max(case when json_extract_scalar(t.x,'$.from_name') like ('%subcategory%')  then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Sub-Category",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'correctint' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Correct information as per SOP / guidelines",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'comprehenddocument' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Was it able to comprehend the uploaded document?",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'allinformation' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Given all information regarding a query (e.g. conveying 3-5 days TAT instead of saying soon)",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'alllqueries' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Addressed all queries of the customer",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'Correctinterpretation' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Correct interpretation of customer's query / message",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'latching' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Not latching to voice of customer (repeating VOC)",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'repetition' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Unique responses by the agent (No word-to-word repititions)",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'hallucinations' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "No hallucinations (by GenAI Bot)",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'clarification' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Asked for customer clarification for any response that it could not understand",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'namerepeat' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Saying customer's name only once (not repeating it)",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'Rushing' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Not Rushing on call",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'Interruption' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Not Interrupting on call",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'jargons' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Replies are easy to understand (no technical terms / jargons)",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'dragging' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Not Dragging the call unneccesarily / active listening",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'unnecessary' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "No Unnecessary information (answer not relevant to questions asked / entertaining random queries not related to Navi)",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'loop' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Did not go on a loop when customer said Thanks / Ok / any other acknowledgement",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'lang' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Adherence to customer language preference",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'understanding' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Demonstrated understanding of customer emotions and concerns",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'furtherassistance' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Seeked further assistance",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'escalationnpci' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Correctly escalated to NPCI/Routed to NPCI",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'closure' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Closing script adherence (Correct CSAT script)",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'opening and closing' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Adhered standard Opening / Closing",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'professional' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Professional behaviour with customer(not referring to customers with slangs like bhai, yaar etc.)",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'escalated agent' then json_format(json_extract(t.x,'$.value.choices')) else null end) as "Correctly escalated to an agent",
    max(case when json_extract_scalar(t.x,'$.from_name') = 'remarks' then json_format(json_extract(t.x,'$.value.text')) else null end) as "Remarks"  
    
    
FROM parsed p
LATERAL VIEW explode(x) t AS t
WHERE date(
    from_unixtime(
        unix_timestamp(
            regexp_replace(substr(annotatedAT, 1, 19), 'T', ' '),
            'yyyy-MM-dd HH:mm:ss'
        ) + 330 * 60
    )
) = current_date()
GROUP BY chat_reference_id, annotatedBy, annotatedAt, result

SELECT DISTINCT a.*, b.csat
FROM data a
LEFT JOIN (
    SELECT 
        canonical_ticket_id,
        AVG(csat) AS csat
    FROM corp_analytics.cx_sub_ticket_data_model_tbl
    GROUP BY canonical_ticket_id
) b 
ON a.chat_reference_id = b.canonical_ticket_id;
"""
df = get_sql_df(query)

# Step 3: Connect to the Google Sheet
gsheet_url = "https://docs.google.com/spreadsheets/d/16f8PrqRpKrLoTHg_hrWllcwx7DhF2sHC4sN9AC6sYRQ/edit?gid=0#gid=0"
worksheet_name = "Voice Bot UPI"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

# Step 4: Clear existing data from the worksheet
gs_reader.clear_sheets(worksheet_name, start_row=1, start_column=2)

# Step 5: Write the new DataFrame to the worksheet (including headers)
gs_reader.set_df_in_sheets(worksheet_name, df, include_headers=True, row=1, col=2)